# Практика: реализация градиентного спуска для линейной регрессии
## Знакомимся

- Иоанн;
- работаю в Яндексе, преподаю в НИУ ВШЭ (и еще где позовут).

## Постановка задачи
Помним, что решаем задачу **линейной регрессии**:

$\hat y = Xw + b$

Нам необходимо подобрать веса таким образом, чтобы предсказание в среднем как можно меньше отличалось от факта. Отличие для одного наблюдения определяем так:

$l = (\hat y - y)^2$

А для всего датасета:

$J(w,b) = \frac{1}{n} \sum_{i=1}^n (\hat y_i - y_i)^2 = \frac{1}{n} \sum_{i=1}^n (x_i^T w + b - y_i)^2$

В матричной записи:

$J(w, b) = \frac{1}{n} * ||X w + b * 1 - y||^2$

Как вы уже знаете из прошлых занятий, можно найти аналитическое решение, а можно "направиться" к желанному минимуму пошагово:

$\nabla J(w) = \frac{2}{n} X^T (X \cdot w - y)$

Еще немного, и получится аналитическое решение! Но нам это сейчас не нужно.

#### Вопрос
В чем, как вам кажется, заключается преимущество GD перед аналитическим решением (матричная запись ниже)?

$w = (X^T X)^{-1} X^T y$

## План поиска минимума
В процессе поиска минимума нам могут встретиться разные проблемы.

## Просто градиентный спуск
Перед тем, как переходить к реализации, давайте подготовим учебные синтетические данные:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)

In [ ]:
# establish constants
N = 2000

# prepare features
x1 = rng.uniform(-3, 3, size=N)
x2 = rng.uniform(-2, 2, size=N) * 10

X = np.column_stack([x1, x2])

# prepare true weights
w_true = np.array([2.5, -8.31])
b_true = 1.2

noise = rng.normal(0, 8.0, size=N)

# prepare y
y = X @ w_true + b_true + noise

# split to train & test
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('w_true:', w_true, 'b_true:', b_true)
print('x1 range:', X[:, 0].min(), X[:, 0].max())
print('x2 range:', X[:, 1].min(), X[:, 1].max())
print('train size:', X_train.shape[0], 'val size:', X_val.shape[0])

Теперь нам нужно применить указанную выше формулу градиента:

$\nabla J(w) = \frac{2}{n} X^T (X \cdot w - y)$

Напишем соответствущий класс. Он должен:

1. Производить обучение модели (искать веса);
2. Уметь предсказывать;
3. Уметь считать качество.

In [ ]:
class LinearRegressionGD:
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.w = None  # надо же с чего-то начинать
        self.b = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)

        # инициализируем веса
        n, d = X.shape

        self.w = np.zeros(d)
        self.b = 0.0

        # шагаем
        for _ in range(self.n_iters):
            y_pred = X @ self.w + self.b
            err = y_pred - y

            grad_w = (2.0 / n) * (X.T @ err)
            grad_b = (2.0 / n) * err.sum()

            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.w + self.b

    def mse(self, X, y):
        y = np.asarray(y, dtype=float).reshape(-1)
        y_pred = self.predict(X)
        return float(np.mean((y_pred - y) ** 2))

Очень хочется скорее применить его на практике. Однако давайте сначала подумаем, удобно ли нам будет пользоваться таким классом. Как мы поймем, что веса найдены? Как мы сможем отслеживать процесс работы?

## Градиентный спуск: инженерные детали
Прежде всего нелишним было бы:

- хранить историю:
    - весов - понимать, насколько мы близки к истине;
    - лосса - понимать, прогрессируем ли мы в обучении;
- выводить прогресс в процессе обучения.

Давайте немного дополним наш класс:

In [ ]:
class LinearRegressionGD:
    def __init__(
        self, lr=0.01, n_iters=1000,

        # настройки логирования
        log_every=10, verbose=False
    ):
        self.lr = lr
        self.n_iters = n_iters
        self.log_every = log_every
        self.verbose = verbose

        self.w = None
        self.b = None

        # для хранения истории
        self.loss_history = []
        self.w_history = []
        self.b_history = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)

        # инициализируем веса
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0

        for t in range(self.n_iters):
            y_pred = X @ self.w + self.b

            # считаем лосс
            err = y_pred - y

            loss = float(np.mean(err ** 2))

            # считаем градиент
            grad_w = (2.0 / n) * (X.T @ err)
            grad_b = (2.0 / n) * err.sum()

            # шагаем
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

            # сохраняем историю
            self.loss_history.append(loss)
            self.w_history.append(self.w.copy())
            self.b_history.append(self.b)

            # логируем
            if self.verbose and (t % self.log_every == 0 or t == self.n_iters - 1):
                print(f'iter={t} loss={loss:.6f} w={self.w} b={self.b:.6f}')

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.w + self.b

    def mse(self, X, y):
        y = np.asarray(y, dtype=float).reshape(-1)
        y_pred = self.predict(X)
        return float(np.mean((y_pred - y) ** 2))

## Градиентный спуск: первый запуск

Применим полученный класс:

In [ ]:
model = LinearRegressionGD(lr=0.001, n_iters=100, log_every=25, verbose=True)
model.fit(X_train, y_train)

In [ ]:
w_true, b_true

Вроде бы, судя по логам, мы продвигаемся неплохо. Посмотрим, что происходило с лоссом на протяжение всего обучения:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(model.loss_history)

ax.set_title('loss decreases during gradient descent')
ax.set_xlabel('iteration')
ax.set_ylabel('mse')

plt.show()

Полюбуемся коэфициентами:

In [ ]:
w_hist = np.array(model.w_history)
b_hist = np.array(model.b_history)

iters = np.arange(w_hist.shape[0])

# рисум
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

axes[0].plot(iters, w_hist[:, 0], label='w1')
axes[0].axhline(w_true[0], linestyle='--', label='w1 true')
axes[0].set_title('convergence of coefficients')
axes[0].set_ylabel('w1')
axes[0].legend()

axes[1].plot(iters, w_hist[:, 1], label='w2')
axes[1].axhline(w_true[1], linestyle='--', label='w2 true')
axes[1].set_ylabel('w2')
axes[1].legend()

axes[2].plot(iters, b_hist, label='b')
axes[2].axhline(b_true, linestyle='--', label='b true')
axes[2].set_xlabel('iteration')
axes[2].set_ylabel('b')
axes[2].legend()

plt.show()

Теперь визуализируем сам спуск. Для этого нам нужно немного постараться. У нас линейная модель:

$\hat y = X_{\text{train\_s}} w + b$

Функция потерь (MSE):

$J(w, b) = \frac{1}{n}\sum_{i=1}^n (\hat y_i - y_i)^2$

Мы хотим нарисовать **контуры** $J$ как функции **двух переменных** $w_1, w_2$. Но у нас ещё есть $b$. Чтобы уложиться в 2d, делаем **срез**: фиксируем $b$ на одном значении и рисуем:

$J_{\text{slice}}(w_1, w_2) = J\big((w_1,w_2), b_{\text{fixed}}\big)$

А затем поверх контуров рисуем точки $(w_1^{(t)}, w_2^{(t)})$, которые получаются во время градиентного спуска.

In [ ]:
w_path = w_hist

# вычисляем границы по траектории, чтобы выбрать область графика
w1_min, w1_max = float(w_path[:, 0].min()), float(w_path[:, 0].max())
w2_min, w2_max = float(w_path[:, 1].min()), float(w_path[:, 1].max())

# добавляем "поля" вокруг траектории, чтобы контуры не обрезались вплотную
pad1 = 0.25 * (w1_max - w1_min + 1e-9)
pad2 = 0.25 * (w2_max - w2_min + 1e-9)

# строим сетку значений w1 и w2
w1_grid = np.linspace(w1_min - pad1, w1_max + pad1, 60)
w2_grid = np.linspace(w2_min - pad2, w2_max + pad2, 60)
W1, W2 = np.meshgrid(w1_grid, w2_grid)

# фиксируем b (делаем 2d-срез)
b_fix = model.b

Z = np.zeros_like(W1)

for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        w_tmp = np.array([W1[i, j], W2[i, j]])
        err = (X_train @ w_tmp + b_fix) - y_train
        Z[i, j] = np.mean(err ** 2)

fig, ax = plt.subplots()
cs = ax.contour(W1, W2, Z, levels=25)
ax.clabel(cs, inline=True, fontsize=8)

ax.plot(w_path[:, 0], w_path[:, 1], marker='o', markersize=2)
ax.scatter(w_path[0, 0], w_path[0, 1], marker='s', s=60, label='start')
ax.scatter(w_path[-1, 0], w_path[-1, 1], marker='*', s=120, label='end')

ax.set_title('gradient descent trajectory on loss contours')
ax.set_xlabel('w1')
ax.set_ylabel('w2')
ax.legend()

plt.show()

Отлично, мы успешно перешли от математики к программированию и даже смогли получить удовлетворительный результат. Впрочем, наш код еще не идеален. Как мы знаем (или догадываемся), существуют различные вариации градиентного спуска (про них мы еще поговорим). Чтобы их реализация было более правильной с инженерной точки зрения, давайте сделаем код более универсальным, а именно обернем в методы отдельные действия:

1. Подсчет лосса;
2. Расчет градиента;
3. Расчет шага;
4. Применение шага.

In [ ]:
class LinearRegressionGD:
    def __init__(
        self, lr=0.01, n_iters=1000, log_every=10, verbose=False
    ):
        # hyperparameters
        self.lr = lr
        self.n_iters = n_iters

        # logging settings
        self.log_every = log_every
        self.verbose = verbose

        # model parameters
        self.w = None
        self.b = None

        # training history
        self.loss_history = []
        self.w_history = []
        self.b_history = []

    def fit(self, X, y):
        # input normalization to predictable numpy shapes
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)

        # initialize model parameters and reset history
        n, d = X.shape
        self._init_params(d)
        self._reset_history()

        # main optimization loop
        for t in range(self.n_iters):
            # forward pass: compute predictions
            y_pred = self.predict(X)

            # compute current loss
            loss = self._compute_loss(y_pred, y)

            # compute gradients of loss w.r.t. parameters
            grad_w, grad_b = self._compute_grad(X, y_pred, y)

            # convert gradients into an actual step in parameter space
            step_w, step_b = self._compute_step(grad_w, grad_b, t)

            # apply update step to parameters
            self._apply_step(step_w, step_b)

            # store training history for plots and debugging
            self._store_history(loss)

            # optional console logging for monitoring
            self._maybe_log(t, loss)

    def predict(self, X):
        # prediction formula for linear model
        X = np.asarray(X, dtype=float)
        return X @ self.w + self.b

    def mse(self, X, y):
        # convenience method to evaluate mse on any dataset
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        y_pred = self.predict(X)
        return self._compute_loss(y_pred, y)

    def _init_params(self, d):
        # initialize weights and bias
        self.w = np.zeros(d)
        self.b = 0.0

    def _reset_history(self):
        # reset stored history (useful if fit is called multiple times)
        self.loss_history = []
        self.w_history = []
        self.b_history = []

    def _compute_loss(self, y_pred, y):
        # compute mse loss
        err = y_pred - y
        return float(np.mean(err ** 2))

    def _compute_grad(self, X, y_pred, y):
        # compute gradients of mse for w and b
        n = X.shape[0]
        err = y_pred - y
        grad_w = (2.0 / n) * (X.T @ err)
        grad_b = (2.0 / n) * err.sum()
        return grad_w, grad_b

    def _compute_step(self, grad_w, grad_b, t):
        # convert gradients into a step (default: plain gradient descent)
        step_w = self.lr * grad_w
        step_b = self.lr * grad_b
        return step_w, step_b

    def _apply_step(self, step_w, step_b):
        # update parameters
        self.w -= step_w
        self.b -= step_b

    def _store_history(self, loss):
        # store values for later visualization
        self.loss_history.append(loss)
        self.w_history.append(self.w.copy())
        self.b_history.append(self.b)

    def _maybe_log(self, t, loss):
        # print progress occasionally
        if self.verbose and (t % self.log_every == 0 or t == self.n_iters - 1):
            print(f'iter={t} loss={loss:.6f} w={self.w} b={self.b:.6f}')

Теперь, когда мы "во всеоружии", можно продолжать путешествие.

## Градиентный спуск: как шагать?
В формуле градиентного спуска обновление выглядит так:


$w \leftarrow w - \eta \nabla J(w), \quad b \leftarrow b - \eta \frac{\partial J}{\partial b}$

Здесь $\eta$ — **learning rate** (часто его называют “шагом”), но важно помнить: фактический размер шага в пространстве параметров равен $\eta \cdot \|\nabla J\|$. поэтому один и тот же $\eta$ может вести себя по-разному в зависимости от масштаба признаков и величины градиента.

В этом блоке мы:
- посмотрим, как меняется поведение обучения при разных значениях `lr` (слишком большой, слишком маленький, "нормальный")
- научимся по графику loss быстро распознавать проблемы (дивергенция, стагнация);
- обсудим, почему разные масштабы признаков могут делать подбор `lr` особенно болезненным и что с этим делать далее.

### Разные lr
Чтобы не дублировать код, логично будет "зашить" построение графиков в сам класс. Возьмем последнюю версию класса и обогатим ее методами для:

1. Построения графика истории лосса;
2. Построения графика истории коэффициентов (по оси на каждый коэффициент).

При этом на графиках должны быть подписаны параметры обучения (lr, количество шагов, etc.)

In [ ]:
class LinearRegressionGD:
    def __init__(
        self, lr=0.01, n_iters=1000, log_every=10, verbose=False
    ):
        # hyperparameters
        self.lr = lr
        self.n_iters = n_iters

        # logging settings
        self.log_every = log_every
        self.verbose = verbose

        # model parameters
        self.w = None
        self.b = None

        # training history
        self.loss_history = []
        self.w_history = []
        self.b_history = []

    def fit(self, X, y):
        # input normalization to predictable numpy shapes
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)

        # initialize model parameters and reset history
        n, d = X.shape
        self._init_params(d)
        self._reset_history()

        # main optimization loop
        for t in range(self.n_iters):
            # forward pass: compute predictions
            y_pred = self.predict(X)

            # compute current loss
            loss = self._compute_loss(y_pred, y)

            # compute gradients of loss w.r.t. parameters
            grad_w, grad_b = self._compute_grad(X, y_pred, y)

            # convert gradients into an actual step in parameter space
            step_w, step_b = self._compute_step(grad_w, grad_b, t)

            # apply update step to parameters
            self._apply_step(step_w, step_b)

            # store training history for plots and debugging
            self._store_history(loss)

            # optional console logging for monitoring
            self._maybe_log(t, loss)

    def predict(self, X):
        # prediction formula for linear model
        X = np.asarray(X, dtype=float)
        return X @ self.w + self.b

    def mse(self, X, y):
        # convenience method to evaluate mse on any dataset
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        y_pred = self.predict(X)
        return self._compute_loss(y_pred, y)

    def plot_loss_history(self, ax=None):  # <---------------------------------------------------- НОВЫЙ МЕТОД
        # plots loss history during training
        if len(self.loss_history) == 0:
            raise ValueError('loss_history is empty, call fit() first')

        title = self._training_title('loss history')
        iters = np.arange(len(self.loss_history))

        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 7))
            created_ax = True
        else:
            fig = ax.figure
            created_ax = False

        ax.plot(iters, self.loss_history)
        ax.set_title(title)
        ax.set_xlabel('iteration')
        ax.set_ylabel('mse')

        fig.tight_layout()
        if created_ax:
            plt.show()

    def plot_coeff_history(self, ax_list=None):  # <---------------------------------------------- НОВЫЙ МЕТОД
        # plots coefficient history on separate axes for each parameter (w1, w2, ..., b)
        if len(self.w_history) == 0 or len(self.b_history) == 0:
            raise ValueError('coefficient history is empty, call fit() first')

        w_hist = np.array(self.w_history)
        b_hist = np.array(self.b_history).reshape(-1)
        iters = np.arange(w_hist.shape[0])

        k = w_hist.shape[1] + 1

        if ax_list is None:
            # default figure size works well for 2 weights + bias; grow height if there are many weights
            fig_height = 7 if k <= 3 else max(7, int(np.ceil(2.2 * k)))
            fig, axes = plt.subplots(k, 1, figsize=(10, fig_height), sharex=True)
            created_axes = True
        else:
            axes = ax_list
            if len(axes) != k:
                raise ValueError(f'expected {k} axes, got {len(axes)}')
            fig = axes[0].figure
            created_axes = False

        title = self._training_title('coefficient convergence')

        if not isinstance(axes, (list, tuple, np.ndarray)):
            axes = [axes]

        axes[0].set_title(title)

        for j in range(w_hist.shape[1]):
            ax = axes[j]
            ax.plot(iters, w_hist[:, j], label=f'w{j + 1}')
            ax.set_ylabel(f'w{j + 1}')
            ax.legend()

        ax_b = axes[-1]
        ax_b.plot(iters, b_hist, label='b')
        ax_b.set_ylabel('b')
        ax_b.set_xlabel('iteration')
        ax_b.legend()

        fig.tight_layout()
        if created_axes:
            plt.show()

    def _init_params(self, d):
        # initialize weights and bias
        self.w = np.zeros(d)
        self.b = 0.0

    def _reset_history(self):
        # reset stored history (useful if fit is called multiple times)
        self.loss_history = []
        self.w_history = []
        self.b_history = []

    def _compute_loss(self, y_pred, y):
        # compute mse loss
        err = y_pred - y
        return float(np.mean(err ** 2))

    def _compute_grad(self, X, y_pred, y):
        # compute gradients of mse for w and b
        n = X.shape[0]
        err = y_pred - y
        grad_w = (2.0 / n) * (X.T @ err)
        grad_b = (2.0 / n) * err.sum()
        return grad_w, grad_b

    def _compute_step(self, grad_w, grad_b, t):
        # convert gradients into a step (default: plain gradient descent)
        step_w = self.lr * grad_w
        step_b = self.lr * grad_b
        return step_w, step_b

    def _apply_step(self, step_w, step_b):
        # update parameters
        self.w -= step_w
        self.b -= step_b

    def _store_history(self, loss):
        # store values for later visualization
        self.loss_history.append(loss)
        self.w_history.append(self.w.copy())
        self.b_history.append(self.b)

    def _maybe_log(self, t, loss):
        # print progress occasionally
        if self.verbose and (t % self.log_every == 0 or t == self.n_iters - 1):
            print(f'iter={t} loss={loss:.6f} w={self.w} b={self.b:.6f}')

    def _training_title(self, base):
        # formats a consistent title with training parameters
        actual_iters = len(self.loss_history) if len(self.loss_history) > 0 else self.n_iters
        return f'{base} (lr={self.lr}, iters={actual_iters})'

Теперь рассмотрим ситуацию для нескольких вариантов lr.

In [ ]:
# 0.001
model = LinearRegressionGD(lr=0.001, n_iters=100, log_every=25, verbose=True)
model.fit(X_train, y_train)

In [ ]:
model.plot_loss_history()

In [ ]:
model.plot_coeff_history()

In [ ]:
# 0.01, явная дивергенция
model = LinearRegressionGD(lr=0.01, n_iters=100, log_every=25, verbose=True)
model.fit(X_train, y_train)

In [ ]:
# 0.00001, явная стагнация
model = LinearRegressionGD(lr=0.00001, n_iters=100, log_every=25, verbose=True)
model.fit(X_train, y_train)

In [ ]:
model.plot_loss_history()

### Проблема разных шкал признаков
Если признаки измеряются в сильно разных масштабах (например, один лежит в диапазоне $[-3, 3]$, а другой в $[-2000, 2000]$), то для линейной регрессии это напрямую влияет на оптимизацию градиентным спуском.

**Интуиция простая:** один и тот же learning rate применяется ко всем параметрам сразу, но вклад разных признаков в градиент может отличаться на порядки. Из-за этого поверхность функции потерь по параметрам становится "вытянутой" (похожа на узкую долину). Тогда градиентный спуск начинает двигаться не напрямую к минимуму, а зигзагом: шаг по одной координате получается слишком большим, по другой — слишком маленьким. в результате приходится уменьшать learning rate ради стабильности, и сходимость становится медленной.

**Главная идея решения:** привести признаки к сопоставимым шкалам (например, стандартизацией), чтобы "долина" стала более округлой, а шаги градиентного спуска — более ровными и быстрыми.

Вспомним график, который мы строили ранее (как всегда, создадим функцию во избежание дублирования кода):

In [ ]:
def plot_descent_contours(
    X, y, w_history,
    b_fix=None, b_history=None,
    ax=None, grid_size=60, levels=25, pad=0.25,
    title=None
):
    # draws loss contours in (w1, w2) and overlays the gd trajectory
    # assumes exactly two features so that w = [w1, w2]
    # uses a 2d slice of the loss surface by fixing b to b_fix (or last value from b_history)

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)

    w_path = np.asarray(w_history, dtype=float)
    if w_path.ndim != 2 or w_path.shape[1] != 2:
        raise ValueError('w_history must have shape (n_steps, 2)')

    if b_fix is None:
        if b_history is not None and len(b_history) > 0:
            b_fix = float(np.asarray(b_history, dtype=float).reshape(-1)[-1])
        else:
            b_fix = 0.0

    w1_min, w1_max = float(w_path[:, 0].min()), float(w_path[:, 0].max())
    w2_min, w2_max = float(w_path[:, 1].min()), float(w_path[:, 1].max())

    pad1 = pad * (w1_max - w1_min + 1e-9)
    pad2 = pad * (w2_max - w2_min + 1e-9)

    w1_grid = np.linspace(w1_min - pad1, w1_max + pad1, grid_size)
    w2_grid = np.linspace(w2_min - pad2, w2_max + pad2, grid_size)
    W1, W2 = np.meshgrid(w1_grid, w2_grid)

    Z = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            w_tmp = np.array([W1[i, j], W2[i, j]])
            err = (X @ w_tmp + b_fix) - y
            Z[i, j] = np.mean(err ** 2)

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))
        created_ax = True
    else:
        fig = ax.figure
        created_ax = False

    cs = ax.contour(W1, W2, Z, levels=levels)
    ax.clabel(cs, inline=True, fontsize=8)

    ax.plot(w_path[:, 0], w_path[:, 1], marker='o', markersize=2, label='trajectory')
    ax.scatter(w_path[0, 0], w_path[0, 1], marker='s', s=70, label='start')
    ax.scatter(w_path[-1, 0], w_path[-1, 1], marker='*', s=140, label='end')

    if title is None:
        title = f'descent trajectory on loss contours (b fixed={b_fix:.3f})'

    ax.set_title(title)
    ax.set_xlabel('w1')
    ax.set_ylabel('w2')
    ax.legend()

    fig.tight_layout()
    if created_ax:
        plt.show()

    return fig, ax

Построим модель с изначальным `lr`:

In [ ]:
model = LinearRegressionGD(lr=0.001, n_iters=100, log_every=25, verbose=True)
model.fit(X_train, y_train)

In [ ]:
_ = plot_descent_contours(
    X_train, y_train,
    w_history=model.w_history,
    b_history=model.b_history,
    title='space: contours + trajectory'
)

Напомним, как его следует понимать:

- по оси X: значение первого веса `w1`;
- по оси Y: значение второго веса `w2`;
- линии (контуры): линии уровня функции потерь `mse(w1, w2)` при фиксированном `b`;
- подписи на линиях: численные значения loss (какой `mse` соответствует этому контуру);
- траектория (линия с точками): последовательность значений (`w1`, `w2`) на итерациях градиентного спуска;
- точка start: стартовые веса (обычно около нулей);
- точка end: финальные веса после `n_iters` итераций;
- расстояние между контурами: насколько быстро меняется loss в этом месте (плотно = "круто", редко = "пологая область").

А теперь применим стандартизацию:

In [ ]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()

X_train_sc = sc.fit_transform(X_train)

In [ ]:
model_sc = LinearRegressionGD(lr=0.001, n_iters=100, log_every=25, verbose=True)
model_sc.fit(X_train_sc, y_train)

In [ ]:
w_true, b_true

In [ ]:
_ = plot_descent_contours(
    X_train_sc, y_train,
    w_history=model_sc.w_history,
    b_history=model_sc.b_history,
    title='space: contours + trajectory'
)

Теперь мы видим, что:

- траектория перестаёт быть **выраженно зигзагообразной** и становится более прямой к минимуму;
- точки итераций располагаются **более равномерно**, без резких "перелётов" через долину;
- при том же lr чаще наблюдается стабильное убывание loss (меньше колебаний).

### Как подобрать `lr`
- начинаем с грубой сетки по порядкам величины: `1e-4, 1e-3, 1e-2, 1e-1, 1` и сужаем диапазон;
- если **loss растёт / сильно колеблется / становится `nan`**, уменьшаем `lr` (проще всего в 10 раз);
- если **loss почти не меняется**, увеличиваем `lr` (в 2–10 раз), но следим за дивергенцией;
- стандартизируем признаки перед подбором `lr`;
- оцениваем `lr` по первым 50–200 итерациям: хороший `lr` даёт быстрое и стабильное снижение.

## Когда мы должны остановиться?
Градиентный спуск — это итерационный алгоритм: он может "улучшать" параметры бесконечно долго, но с практической точки зрения нам нужно заранее определить момент остановки. Критерий остановки отвечает на вопрос: **какой прогресс считать достаточным** и **когда дальнейшие итерации уже не дают смысла** (или становятся слишком дорогими по времени).

В этом блоке мы:
- разберём основные варианты критериев остановки (по числу итераций и по изменению loss);
- реализуем их в коде максимально просто;
- посмотрим на графиках, как разные критерии влияют на итоговые параметры и стабильность обучения.

Собственно говоря, остановку по числу итераций мы уже реализовали. Теперь посмотрим на вариант с изменением loss. Стратегия тут простая: останавливаем обучение, когда loss почти перестал уменьшаться.

**Типичный критерий**: если за шаг (или за несколько последних шагов) улучшение меньше порога

$|L_{t-1} - L_t| < \varepsilon$

При этом надо понимать, что шум может заставить алгоритм остановиться раньше необходимого. Поэтому сделаем так:

1. Укажем минимальное количество итераций;
2. Максимальное количество итераций;
3. В промежутке будем останавливаться по указанному критерию.

In [ ]:
class LinearRegressionGD:
    def __init__(
        self,
        lr=0.01,
        n_iters=1000,
        min_iters=50,
        tol=1e-6,
        log_every=10,
        verbose=False,
    ):
        # hyperparameters
        self.lr = lr
        self.n_iters = n_iters

        # stopping settings
        self.min_iters = min_iters
        self.tol = tol

        # logging settings
        self.log_every = log_every
        self.verbose = verbose

        # model parameters
        self.w = None
        self.b = None

        # training history
        self.loss_history = []
        self.w_history = []
        self.b_history = []

        # fitted meta
        self.n_iters_run = 0
        self.stop_reason = None

    def fit(self, X, y):
        # input normalization to predictable numpy shapes
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)

        # initialize model parameters and reset history
        n, d = X.shape
        self._init_params(d)
        self._reset_history()

        # reset fit meta
        self.n_iters_run = 0
        self.stop_reason = None

        prev_loss = None

        # main optimization loop
        for t in range(self.n_iters):
            # forward pass: compute predictions
            y_pred = self.predict(X)

            # compute current loss
            loss = self._compute_loss(y_pred, y)

            # compute gradients of loss w.r.t. parameters
            grad_w, grad_b = self._compute_grad(X, y_pred, y)

            # convert gradients into an actual step in parameter space
            step_w, step_b = self._compute_step(grad_w, grad_b, t)

            # apply update step to parameters
            self._apply_step(step_w, step_b)

            # store training history for plots and debugging
            self._store_history(loss)

            # optional console logging for monitoring
            self._maybe_log(t, loss)

            # stopping strategy: min iters + max iters + stop by loss change
            if prev_loss is not None and t + 1 >= self.min_iters:
                if abs(prev_loss - loss) < self.tol:
                    self.n_iters_run = t + 1
                    self.stop_reason = 'loss_change_below_tol'
                    break

            prev_loss = loss

            self.n_iters_run = t + 1

        if self.stop_reason is None:
            self.stop_reason = 'max_iters_reached'

    def predict(self, X):
        # prediction formula for linear model
        X = np.asarray(X, dtype=float)
        return X @ self.w + self.b

    def mse(self, X, y):
        # convenience method to evaluate mse on any dataset
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        y_pred = self.predict(X)
        return self._compute_loss(y_pred, y)

    def plot_loss_history(self, ax=None):
        # plots loss history during training
        if len(self.loss_history) == 0:
            raise ValueError('loss_history is empty, call fit() first')

        title = self._training_title('loss history')
        iters = np.arange(len(self.loss_history))

        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 7))
            created_ax = True
        else:
            fig = ax.figure
            created_ax = False

        ax.plot(iters, self.loss_history)
        ax.set_title(title)
        ax.set_xlabel('iteration')
        ax.set_ylabel('mse')

        fig.tight_layout()
        if created_ax:
            plt.show()

    def plot_coeff_history(self, ax_list=None):
        # plots coefficient history on separate axes for each parameter (w1, w2, ..., b)
        if len(self.w_history) == 0 or len(self.b_history) == 0:
            raise ValueError('coefficient history is empty, call fit() first')

        w_hist = np.array(self.w_history)
        b_hist = np.array(self.b_history).reshape(-1)
        iters = np.arange(w_hist.shape[0])

        k = w_hist.shape[1] + 1

        if ax_list is None:
            fig_height = 7 if k <= 3 else max(7, int(np.ceil(2.2 * k)))
            fig, axes = plt.subplots(k, 1, figsize=(10, fig_height), sharex=True)
            created_axes = True
        else:
            axes = ax_list
            if len(axes) != k:
                raise ValueError(f'expected {k} axes, got {len(axes)}')
            fig = axes[0].figure
            created_axes = False

        title = self._training_title('coefficient convergence')

        if not isinstance(axes, (list, tuple, np.ndarray)):
            axes = [axes]

        axes[0].set_title(title)

        for j in range(w_hist.shape[1]):
            ax = axes[j]
            ax.plot(iters, w_hist[:, j], label=f'w{j + 1}')
            ax.set_ylabel(f'w{j + 1}')
            ax.legend()

        ax_b = axes[-1]
        ax_b.plot(iters, b_hist, label='b')
        ax_b.set_ylabel('b')
        ax_b.set_xlabel('iteration')
        ax_b.legend()

        fig.tight_layout()
        if created_axes:
            plt.show()

    def _init_params(self, d):
        # initialize weights and bias
        self.w = np.zeros(d)
        self.b = 0.0

    def _reset_history(self):
        # reset stored history (useful if fit is called multiple times)
        self.loss_history = []
        self.w_history = []
        self.b_history = []

    def _compute_loss(self, y_pred, y):
        # compute mse loss
        err = y_pred - y
        return float(np.mean(err ** 2))

    def _compute_grad(self, X, y_pred, y):
        # compute gradients of mse for w and b
        n = X.shape[0]
        err = y_pred - y
        grad_w = (2.0 / n) * (X.T @ err)
        grad_b = (2.0 / n) * err.sum()
        return grad_w, grad_b

    def _compute_step(self, grad_w, grad_b, t):
        # convert gradients into a step (default: plain gradient descent)
        step_w = self.lr * grad_w
        step_b = self.lr * grad_b
        return step_w, step_b

    def _apply_step(self, step_w, step_b):
        # update parameters
        self.w -= step_w
        self.b -= step_b

    def _store_history(self, loss):
        # store values for later visualization
        self.loss_history.append(loss)
        self.w_history.append(self.w.copy())
        self.b_history.append(self.b)

    def _maybe_log(self, t, loss):
        # print progress occasionally
        if self.verbose and (t % self.log_every == 0 or t == self.n_iters - 1):
            print(f'iter={t} loss={loss:.6f} w={self.w} b={self.b:.6f}')

    def _training_title(self, base):
        # formats a consistent title with training parameters
        actual_iters = len(self.loss_history) if len(self.loss_history) > 0 else self.n_iters
        reason = self.stop_reason if self.stop_reason is not None else 'not_fitted'
        return f'{base} (lr={self.lr}, iters={actual_iters}, min_iters={self.min_iters}, tol={self.tol}, stop={reason})'

Теперь применим:

In [ ]:
model = LinearRegressionGD(lr=0.001, log_every=25, verbose=True)
model.fit(X_train, y_train)

In [ ]:
model.plot_coeff_history()

In [ ]:
_ = plot_descent_contours(
    X_train, y_train,
    w_history=model.w_history,
    b_history=model.b_history,
    title='space: contours + trajectory'
)

#### Вопрос
Что можно сделать с таким солидным зигзагом?

In [ ]:
# наш код здесь

## Количество наблюдений
В каждом шаге градиентного спуска нам нужен градиент функции потерь. Вопрос в том, **по скольким наблюдениям** его считать:

- по всей выборке сразу (batch gradient descent);
- по одному наблюдению (single gradient descent);
- по небольшому подмножеству (mini-batch gradient descent).

В этом блоке мы:
- сравним эти варианты на одной и той же задаче и посмотрим, как меняется график loss;
- обсудим компромисс "точность градиента vs скорость шага": полный градиент точнее, но дороже, а sgd дешевле, но шумнее;
- добавим в код поддержку `batch_size`, чтобы переключаться между режимами одной настройкой;
- разберём, почему при sgd/mini-batch история loss часто становится "рваной" и как это учитывать при выборе learning rate и критериев остановки.

#### Задание
Добавьте в код поддержку `batch_size` и сравните, как меняется график loss в зависимости от размера батча.

In [ ]:
# наш код здесь